In [1]:
import pandas as pd

# ------------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------------
input_file = "nationaldatabaseofchildcareprices.xlsx"
output_file = "state_level_childcare_metrics.xlsx"

df = pd.read_excel(input_file)

In [2]:
# ------------------------------------------------------------------
# 1. Median MHI per State per StudyYear → StateMHI
# ------------------------------------------------------------------
state_mhi = (
    df.groupby(["State_Name", "StudyYear"], as_index=False)
      .agg(StateMHI=("MHI", "median"))
)

In [3]:
# ------------------------------------------------------------------
# 2. Median StateMHI across all states (overall reference value)
# ------------------------------------------------------------------
median_state_mhi_all = state_mhi["StateMHI"].median()

# ------------------------------------------------------------------
# 3. Median childcare prices per State per StudyYear
# ------------------------------------------------------------------
state_childcare = (
    df.groupby(["State_Name", "StudyYear"], as_index=False)
      .agg(
          State_MFCCSA=("MFCCSA", "median"),
          State_MFCCInfant=("MFCCInfant", "median"),
          State_MFCCToddler=("MFCCToddler", "median"),
          State_MFCCPreschool=("MFCCPreschool", "median")
      )
)

# ------------------------------------------------------------------
# Convert weekly childcare costs to annual (multiply by 52)
# ------------------------------------------------------------------
annual_cost_vars = [
    "State_MFCCSA",
    "State_MFCCInfant",
    "State_MFCCToddler",
    "State_MFCCPreschool"
]

state_childcare[annual_cost_vars] = state_childcare[annual_cost_vars] * 52

# ------------------------------------------------------------------
# 4. Median MFCCSA across all states for StudyYear == 2018
#    divided by Median StateMHI
# ------------------------------------------------------------------
mfccsa_2018 = (
    state_childcare[state_childcare["StudyYear"] == 2018]["State_MFCCSA"]
    .median()
)

mfccsa_to_mhi_ratio_2018 = mfccsa_2018 / median_state_mhi_all

# ------------------------------------------------------------------
# 5. Median PR_F per State per StudyYear → StatePR_F
# ------------------------------------------------------------------
state_poverty = (
    df.groupby(["State_Name", "StudyYear"], as_index=False)
      .agg(StatePR_F=("PR_F", "median"))
)

# ------------------------------------------------------------------
# 6. % increase in annualized Median MFCCSA
#    from earliest to latest available StudyYear per state
# ------------------------------------------------------------------

# Keep only rows with valid MFCCSA
mfccsa_valid = state_childcare[
    state_childcare["State_MFCCSA"].notna()
].sort_values(["State_Name", "StudyYear"])

# Earliest non-null MFCCSA per state
earliest_mfccsa = (
    mfccsa_valid
    .groupby("State_Name", as_index=False)
    .first()[["State_Name", "StudyYear", "State_MFCCSA"]]
    .rename(columns={
        "StudyYear": "EarliestYear",
        "State_MFCCSA": "MFCCSA_Earliest"
    })
)

# Latest non-null MFCCSA per state
latest_mfccsa = (
    mfccsa_valid
    .groupby("State_Name", as_index=False)
    .last()[["State_Name", "StudyYear", "State_MFCCSA"]]
    .rename(columns={
        "StudyYear": "LatestYear",
        "State_MFCCSA": "MFCCSA_Latest"
    })
)

# Merge earliest and latest records
mfccsa_change = earliest_mfccsa.merge(
    latest_mfccsa,
    on="State_Name",
    how="inner"
)

# Calculate percent increase
mfccsa_change["MFCCSA_pct_increase"] = (
    (mfccsa_change["MFCCSA_Latest"] - mfccsa_change["MFCCSA_Earliest"])
    / mfccsa_change["MFCCSA_Earliest"]
) * 100

# ------------------------------------------------------------------
# 7. Merge all state-level metrics into one dataset
# ------------------------------------------------------------------
final_df = (
    state_mhi
    .merge(state_childcare, on=["State_Name", "StudyYear"], how="left")
    .merge(state_poverty, on=["State_Name", "StudyYear"], how="left")
)

final_df = final_df.merge(
    mfccsa_change[
        [
            "State_Name",
            "EarliestYear",
            "LatestYear",
            "MFCCSA_pct_increase"
        ]
    ],
    on="State_Name",
    how="left"
)

In [4]:
final_df.head()

,State_Name,StudyYear,StateMHI,State_MFCCSA,State_MFCCInfant,State_MFCCToddler,State_MFCCPreschool,StatePR_F,EarliestYear,LatestYear,MFCCSA_pct_increase
0,Alabama,2008,34952.33,4232.80,4339.40,4339.40,4232.80,13.9,2008.0,2018.0,17.862408
1,Alabama,2009,35137.00,4455.36,4544.28,4544.28,4455.36,14.0,2008.0,2018.0,17.862408
2,Alabama,2010,36077.00,4677.92,4749.16,4749.16,4677.92,14.7,2008.0,2018.0,17.862408
3,Alabama,2011,37161.00,4901.00,4954.56,4954.56,4901.00,15.4,2008.0,2018.0,17.862408
4,Alabama,2012,37059.00,4929.08,5034.64,5034.64,4929.08,15.4,2008.0,2018.0,17.862408


In [6]:
final_df.to_excel(
    output_file,
    sheet_name="State_Level_Metrics",
    index=False
)

print("State-level dataset successfully created and saved.")
print(f"Median StateMHI (all states): {median_state_mhi_all:,.2f}")
print(f"2018 MFCCSA / Median StateMHI ratio: {mfccsa_to_mhi_ratio_2018:.3f}")

State-level dataset successfully created and saved.
Median StateMHI (all states): 46,872.00
2018 MFCCSA / Median StateMHI ratio: 0.114
